# 3. Decision Trees

**Machine Learning Fundamentals and Predictive Analytics — Notebook 3 of 11**

A **decision tree** learns a flowchart. *Is tenure below 12 months? If yes, is the contract
monthly? If yes, predict churn.* It is the only mainstream model that a non-technical
stakeholder can read directly off the page — which is why trees survive in credit scoring,
medicine and operations long after fancier models exist.

They are also the building block of the two strongest tabular algorithms in practice: random
forests (Notebook 6) and gradient boosting. Understand the tree and you understand both.

### What you will learn

1. How a tree splits: **Gini impurity**, **entropy**, **information gain**
2. Building a tree by hand, one split at a time
3. Growing and reading one with scikit-learn
4. **Regression trees** and the variance-reduction criterion
5. Why trees overfit, and how to **prune** them
6. **Feature importance** — and its biases
7. The geometry: axis-aligned rectangles, and what that costs
8. Strengths, weaknesses, and when to reach for a tree

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import (DecisionTreeClassifier, DecisionTreeRegressor, plot_tree,
                          export_text)
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     StratifiedKFold, KFold, validation_curve)
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             mean_squared_error, r2_score, roc_auc_score)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

rng = np.random.default_rng(seed=3)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)
CV = KFold(5, shuffle=True, random_state=0)

---
## 3.1 The idea

A tree repeatedly asks a yes/no question about **one feature at a time**, splitting the data
into ever-purer groups.

- **Root node** — all the data
- **Internal node** — a test, e.g. `tenure <= 11.5`
- **Branch** — the yes/no outcome
- **Leaf** — a terminal node holding a prediction (majority class, or mean value)
- **Depth** — the longest path from root to leaf

Prediction is trivial: walk the tree, answer the questions, read the leaf.

The learning problem is: **which question, at which node?** The answer is greedy — at each
node, try every feature and every threshold, and take the split that most reduces impurity.
This is not globally optimal (finding the optimal tree is NP-hard), but it works well and
runs fast.

---
## 3.2 Measuring impurity

A node is **pure** if all its samples share one class. We need a number for "how mixed is
this?" Two are used, where $p_k$ is the proportion of class $k$ in the node:

**Gini impurity** — the probability of misclassifying a random sample if you labelled it
by drawing from the node's class distribution:

$$G = 1 - \sum_{k} p_k^2$$

**Entropy** — the information-theoretic uncertainty, in bits:

$$H = -\sum_{k} p_k \log_2 p_k$$

Both are 0 for a pure node and maximal for a perfectly mixed one (for two classes: $G = 0.5$,
$H = 1$ bit at $p = 0.5$). They give nearly identical trees; Gini is the scikit-learn default
because it avoids a logarithm.

In [ ]:
def gini(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return 1 - (p ** 2).sum()

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

print(f"{'class mix':>22}{'Gini':>10}{'entropy':>10}")
for mix in [[1.0, 0.0], [0.9, 0.1], [0.7, 0.3], [0.5, 0.5],
            [1/3, 1/3, 1/3], [0.25]*4]:
    label = "[" + ", ".join(f"{v:.2f}" for v in mix) + "]"
    print(f"{label:>22}{gini(mix):>10.4f}{entropy(mix):>10.4f}")

p1 = np.linspace(0.001, 0.999, 400)
plt.plot(p1, [gini([p_, 1-p_]) for p_ in p1], lw=2.4, color="steelblue", label="Gini")
plt.plot(p1, [entropy([p_, 1-p_]) for p_ in p1], lw=2.4, color="crimson", label="entropy (bits)")
plt.plot(p1, [min(p_, 1-p_) for p_ in p1], lw=1.6, ls="--", color="grey",
         label="misclassification rate")
plt.axvline(0.5, color="black", ls=":", lw=1)
plt.xlabel("proportion of class 1"); plt.ylabel("impurity")
plt.title("Impurity is maximal at a 50/50 mix and zero at the extremes")
plt.legend(fontsize=8); plt.show()

### Information gain

The value of a split is the impurity you remove, weighted by how many samples go each way:

$$\text{Gain} = I(\text{parent}) - \sum_{\text{child } j}\frac{n_j}{n}\,I(\text{child}_j)$$

The tree picks the (feature, threshold) pair with the largest gain. Note the weighting: a
split that produces one tiny pure leaf and one large mixed leaf scores poorly, which is what
stops the tree from peeling off single points.

In [ ]:
# A tiny dataset we can work through by hand
data = pd.DataFrame({
    "age":      [22, 25, 27, 35, 38, 41, 46, 52, 55, 58, 61, 65],
    "income":   [18, 22, 35, 45, 30, 60, 52, 75, 40, 82, 55, 90],   # thousands
    "buys":     [0,  0,  0,  1,  0,  1,  1,  1,  0,  1,  1,  1],
})
print(data.to_string(index=False))

n_tot = len(data)
p_parent = data.buys.value_counts(normalize=True).sort_index().to_numpy()
print(f"\nParent node: {data.buys.sum()} buyers of {n_tot}")
print(f"  Gini    = {gini(p_parent):.4f}")
print(f"  entropy = {entropy(p_parent):.4f} bits")

In [ ]:
def split_gain(frame, feature, threshold, target="buys", criterion=gini):
    '''Weighted impurity reduction for a single candidate split.'''
    left = frame[frame[feature] <= threshold][target]
    right = frame[frame[feature] > threshold][target]
    if len(left) == 0 or len(right) == 0:
        return None
    n = len(frame)
    imp_parent = criterion(frame[target].value_counts(normalize=True))
    imp_left = criterion(left.value_counts(normalize=True))
    imp_right = criterion(right.value_counts(normalize=True))
    weighted = len(left)/n * imp_left + len(right)/n * imp_right
    return {"feature": feature, "threshold": threshold,
            "n_left": len(left), "n_right": len(right),
            "imp_left": imp_left, "imp_right": imp_right,
            "weighted_child_impurity": weighted, "gain": imp_parent - weighted}

# Candidate thresholds are the midpoints between consecutive sorted values
candidates = []
for feat in ["age", "income"]:
    vals = np.sort(data[feat].unique())
    for a, b in zip(vals[:-1], vals[1:]):
        g = split_gain(data, feat, (a + b) / 2)
        if g:
            candidates.append(g)

cand = pd.DataFrame(candidates).sort_values("gain", ascending=False)
print("Top 8 candidate splits by information gain (Gini):")
print(cand.head(8).round(4).to_string(index=False))
best = cand.iloc[0]
print(f"\nBEST SPLIT: {best.feature} <= {best.threshold}  (gain {best.gain:.4f})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for feat, a_ in zip(["age", "income"], ax):
    sub = cand[cand.feature == feat].sort_values("threshold")
    a_.plot(sub.threshold, sub.gain, "o-", color="steelblue")
    top = sub.loc[sub.gain.idxmax()]
    a_.axvline(top.threshold, color="crimson", ls="--",
               label=f"best: {feat} <= {top.threshold:.1f} (gain {top.gain:.3f})")
    a_.set_xlabel(f"threshold on {feat}"); a_.set_ylabel("information gain")
    a_.set_title(f"Every candidate split on {feat}", fontsize=10); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("This exhaustive scan is exactly what the tree does at every node --")
print("for every feature, at every possible threshold. It is greedy but cheap.")

In [ ]:
# Now recurse, by hand, one level down
left = data[data[best.feature] <= best.threshold]
right = data[data[best.feature] > best.threshold]
print(f"After splitting on {best.feature} <= {best.threshold}:")
print(f"  LEFT  n={len(left):>2}, buyers={left.buys.sum()}, "
      f"Gini={gini(left.buys.value_counts(normalize=True)):.4f}")
print(f"  RIGHT n={len(right):>2}, buyers={right.buys.sum()}, "
      f"Gini={gini(right.buys.value_counts(normalize=True)):.4f}\n")

for name, node in [("LEFT", left), ("RIGHT", right)]:
    if node.buys.nunique() == 1:
        print(f"  {name} is already pure -> it becomes a leaf predicting {node.buys.iloc[0]}")
        continue
    sub_cands = []
    for feat in ["age", "income"]:
        vals = np.sort(node[feat].unique())
        for a, b in zip(vals[:-1], vals[1:]):
            g = split_gain(node, feat, (a + b) / 2)
            if g:
                sub_cands.append(g)
    if sub_cands:
        b2 = pd.DataFrame(sub_cands).sort_values("gain", ascending=False).iloc[0]
        print(f"  {name} best next split: {b2.feature} <= {b2.threshold} "
              f"(gain {b2.gain:.4f})")

---
## 3.3 The same tree with scikit-learn

Now let the library do it, and check that it agrees with our hand calculation.

In [ ]:
X_toy, y_toy = data[["age", "income"]], data["buys"]
tree = DecisionTreeClassifier(criterion="gini", random_state=0).fit(X_toy, y_toy)

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree, feature_names=["age", "income"], class_names=["no", "buys"],
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.title("The fitted tree")
plt.show()

print(export_text(tree, feature_names=["age", "income"]))
print(f"Root split chosen by sklearn : "
      f"{['age','income'][tree.tree_.feature[0]]} <= {tree.tree_.threshold[0]:.2f}")
print(f"Root split we computed       : {best.feature} <= {best.threshold:.2f}")
print(f"Depth {tree.get_depth()}, {tree.get_n_leaves()} leaves, "
      f"training accuracy {tree.score(X_toy, y_toy):.3f}")

### How to read the diagram

Each box shows:

- the **test** (absent in leaves)
- **gini** — the node's impurity
- **samples** — how many training rows reach it
- **value** — the class counts, `[n_class0, n_class1]`
- **class** — the majority class, which is the prediction

The colour intensity encodes purity. A prediction is just the leaf's majority class; the
**predicted probability** is the class proportion in that leaf.

In [ ]:
# Following a single prediction through the tree
sample = pd.DataFrame({"age": [44], "income": [58]})
print(f"Predicting for age=44, income=58:")
print(f"  predicted class       : {tree.predict(sample)[0]}")
print(f"  predicted probability : {tree.predict_proba(sample)[0]}")
leaf_id = tree.apply(sample)[0]
print(f"  lands in leaf node id : {leaf_id}")
print(f"  that leaf's training counts: {tree.tree_.value[leaf_id][0]}")
print()
print("Decision path:")
node_indicator = tree.decision_path(sample)
feature_names = ["age", "income"]
for node in node_indicator.indices:
    if tree.tree_.children_left[node] == -1:
        print(f"  node {node}: LEAF -> predict {tree.classes_[np.argmax(tree.tree_.value[node][0])]}")
    else:
        f_ = feature_names[tree.tree_.feature[node]]
        t_ = tree.tree_.threshold[node]
        went = "left (yes)" if sample[f_].item() <= t_ else "right (no)"
        print(f"  node {node}: is {f_} <= {t_:.2f}?  {went}")

---
## 3.4 Regression trees

The same algorithm, a different impurity. For regression the tree minimises **squared error**
within each node — equivalently, it maximises the reduction in variance:

$$I(\text{node}) = \frac{1}{n}\sum_{i\in\text{node}}(y_i - \bar{y}_{\text{node}})^2$$

Each leaf predicts the **mean** of its training targets. This means a regression tree produces
a **piecewise-constant** function: a staircase, not a curve.

In [ ]:
xs = np.sort(rng.uniform(0, 10, 120))
ys = np.sin(xs) + 0.25*xs + rng.normal(0, 0.3, 120)
Xr = xs.reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
grid = np.linspace(0, 10, 600).reshape(-1, 1)
for ax, depth in zip(axes, [1, 2, 4, None]):
    t = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(Xr, ys)
    ax.scatter(xs, ys, s=16, alpha=0.6, color="steelblue")
    ax.plot(grid, t.predict(grid), color="crimson", lw=2)
    ax.set_title(f"max_depth = {depth}\n{t.get_n_leaves()} leaves, "
                 f"train R^2 = {t.score(Xr, ys):.3f}", fontsize=9)
plt.tight_layout(); plt.show()

print("Trees approximate a smooth curve with steps. More depth = more, narrower steps.")
print("The unlimited tree has one leaf per training point: R^2 = 1.000 and no generalisation.")
print("\nA crucial consequence: a tree CANNOT EXTRAPOLATE. Outside the training range it")
print("returns the nearest leaf's constant, forever.")

In [ ]:
# Demonstrating the extrapolation failure
x_ext = np.linspace(0, 20, 400).reshape(-1, 1)
t4 = DecisionTreeRegressor(max_depth=4, random_state=0).fit(Xr, ys)
from sklearn.linear_model import LinearRegression
lin_ext = LinearRegression().fit(Xr, ys)

plt.scatter(xs, ys, s=16, color="steelblue", label="training data (x in 0-10)")
plt.plot(x_ext, t4.predict(x_ext), color="crimson", lw=2, label="decision tree")
plt.plot(x_ext, lin_ext.predict(x_ext), color="seagreen", lw=2, label="linear regression")
plt.plot(x_ext, np.sin(x_ext.ravel()) + 0.25*x_ext.ravel(), "k--", lw=1.2, label="truth")
plt.axvline(10, color="black", lw=1)
plt.legend(fontsize=8); plt.title("Beyond the data, the tree is flat forever")
plt.show()

print(f"Tree prediction at x=12 : {t4.predict([[12]])[0]:.3f}")
print(f"Tree prediction at x=20 : {t4.predict([[20]])[0]:.3f}  <- identical")
print(f"True value at x=20      : {np.sin(20) + 5:.3f}")
print("\nIf your problem needs extrapolation (trends, growth), do not use a bare tree.")

---
## 3.5 Why trees overfit, and how to control them

An unconstrained tree keeps splitting until every leaf is pure — which means memorising the
training set. The regularisation knobs:

| Parameter | Effect |
|---|---|
| `max_depth` | Hard cap on tree height. The bluntest, most effective knob |
| `min_samples_split` | A node must have at least this many samples to be split |
| `min_samples_leaf` | Every leaf must keep at least this many samples — directly prevents peeling off individual points |
| `max_leaf_nodes` | Cap the total number of leaves (best-first growth) |
| `min_impurity_decrease` | Reject splits that gain less than this |
| `ccp_alpha` | **Cost-complexity pruning**: grow fully, then prune back |
| `max_features` | Consider only a random subset of features per split (the core idea of random forests) |

**Pre-pruning** (the first six) stops growth early. **Post-pruning** (`ccp_alpha`) grows the
full tree then removes subtrees that do not pay for themselves — usually the better approach,
because a split that looks useless can enable a good one below it.

In [ ]:
# A realistic classification dataset
m = 2_000
tenure = rng.exponential(20, m).clip(0, 72)
monthly = rng.normal(65, 20, m).clip(15, 130)
support = rng.poisson(1.3, m)
n_services = rng.integers(1, 7, m)
z = -1.2 - 0.05*tenure + 0.018*monthly + 0.34*support - 0.12*n_services
churn = (rng.random(m) < 1/(1+np.exp(-z))).astype(int)

cust = pd.DataFrame({"tenure": tenure, "monthly": monthly, "support": support,
                     "n_services": n_services, "churn": churn})
Xc, yc = cust.drop(columns="churn"), cust["churn"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0,
                                              stratify=yc)
print(f"{m} customers, churn rate {yc.mean():.3f}")
print(f"Baseline accuracy: "
      f"{DummyClassifier(strategy='most_frequent').fit(Xc_tr, yc_tr).score(Xc_te, yc_te):.4f}")

In [ ]:
depths = list(range(1, 26))
tr_s, va_s = validation_curve(DecisionTreeClassifier(random_state=0), Xc_tr, yc_tr,
                              param_name="max_depth", param_range=depths,
                              cv=SKF, scoring="roc_auc")
plt.plot(depths, tr_s.mean(1), "o-", color="steelblue", label="training ROC-AUC")
plt.plot(depths, va_s.mean(1), "o-", color="crimson", label="cross-validated ROC-AUC")
best_depth = depths[int(np.argmax(va_s.mean(1)))]
plt.axvline(best_depth, color="black", ls="--", label=f"best depth = {best_depth}")
plt.xlabel("max_depth"); plt.ylabel("ROC-AUC")
plt.title("Trees overfit fast: training AUC goes to 1, CV AUC peaks early")
plt.legend(fontsize=8); plt.show()

print(f"{'depth':>7}{'train AUC':>12}{'CV AUC':>10}")
for d, a, b_ in zip(depths, tr_s.mean(1), va_s.mean(1)):
    if d <= 10 or d % 5 == 0:
        print(f"{d:>7}{a:>12.4f}{b_:>10.4f}{'  <- best' if d == best_depth else ''}")

In [ ]:
# Cost-complexity pruning: grow fully, then prune
full = DecisionTreeClassifier(random_state=0).fit(Xc_tr, yc_tr)
path = full.cost_complexity_pruning_path(Xc_tr, yc_tr)
alphas = path.ccp_alphas[:-1]                      # drop the alpha that collapses to one node
alphas = alphas[alphas > 0][::max(1, len(alphas)//40)]

rows = []
for a_ in alphas:
    t = DecisionTreeClassifier(ccp_alpha=a_, random_state=0)
    cvv = cross_val_score(t, Xc_tr, yc_tr, cv=SKF, scoring="roc_auc").mean()
    t.fit(Xc_tr, yc_tr)
    rows.append({"ccp_alpha": a_, "leaves": t.get_n_leaves(), "depth": t.get_depth(),
                 "train_auc": roc_auc_score(yc_tr, t.predict_proba(Xc_tr)[:, 1]),
                 "cv_auc": cvv})
pr = pd.DataFrame(rows)
best_row = pr.loc[pr.cv_auc.idxmax()]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(pr.ccp_alpha, pr.train_auc, "o-", color="steelblue", label="training")
ax[0].plot(pr.ccp_alpha, pr.cv_auc, "o-", color="crimson", label="cross-validated")
ax[0].axvline(best_row.ccp_alpha, color="black", ls="--")
ax[0].set_xscale("log"); ax[0].set_xlabel("ccp_alpha"); ax[0].set_ylabel("ROC-AUC")
ax[0].set_title("Cost-complexity pruning"); ax[0].legend(fontsize=8)
ax[1].plot(pr.ccp_alpha, pr.leaves, "o-", color="seagreen")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("ccp_alpha"); ax[1].set_ylabel("number of leaves")
ax[1].set_title("Bigger alpha = smaller tree")
plt.tight_layout(); plt.show()

print(f"Full tree     : {full.get_n_leaves()} leaves, depth {full.get_depth()}, "
      f"test AUC {roc_auc_score(yc_te, full.predict_proba(Xc_te)[:, 1]):.4f}")
pruned = DecisionTreeClassifier(ccp_alpha=best_row.ccp_alpha, random_state=0).fit(Xc_tr, yc_tr)
print(f"Pruned tree   : {pruned.get_n_leaves()} leaves, depth {pruned.get_depth()}, "
      f"test AUC {roc_auc_score(yc_te, pruned.predict_proba(Xc_te)[:, 1]):.4f}")
print(f"Best ccp_alpha = {best_row.ccp_alpha:.6f}")

In [ ]:
# Tune several knobs together
gs = GridSearchCV(DecisionTreeClassifier(random_state=0),
                  {"max_depth": [3, 4, 5, 6, 8, 10, None],
                   "min_samples_leaf": [1, 5, 10, 25, 50],
                   "criterion": ["gini", "entropy"]},
                  cv=SKF, scoring="roc_auc", n_jobs=1).fit(Xc_tr, yc_tr)

print(f"Best parameters : {gs.best_params_}")
print(f"Best CV ROC-AUC : {gs.best_score_:.4f}")
print(f"Test ROC-AUC    : {roc_auc_score(yc_te, gs.predict_proba(Xc_te)[:, 1]):.4f}")
print(f"Test accuracy   : {accuracy_score(yc_te, gs.predict(Xc_te)):.4f}\n")

# Gini vs entropy: usually a wash
res = pd.DataFrame(gs.cv_results_)
print("Mean CV AUC by criterion:")
print(res.groupby("param_criterion")["mean_test_score"].agg(["mean", "max"]).round(4).to_string())
print("\nAs promised: the choice of impurity measure barely matters. Depth and leaf size do.")

In [ ]:
# The tuned tree, small enough to read
final = DecisionTreeClassifier(max_depth=3, min_samples_leaf=25, random_state=0).fit(Xc_tr, yc_tr)
fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(final, feature_names=list(Xc.columns), class_names=["stay", "churn"],
          filled=True, rounded=True, fontsize=8, ax=ax, proportion=True)
plt.title("A readable churn tree")
plt.show()

print(export_text(final, feature_names=list(Xc.columns), decimals=1))
print("This is a business rulebook, not a black box. That is the selling point of trees.")

---
## 3.6 Feature importance

Trees report which features they used. Two definitions, and the difference matters.

**Impurity-based (`feature_importances_`)** — total impurity reduction contributed by each
feature, weighted by the samples reaching each split. Free, but **biased**: it inflates
high-cardinality and continuous features because they offer more candidate thresholds, and it
is computed on the *training* data.

**Permutation importance** — shuffle one column in the **validation** data and measure how
much performance drops. Slower, but it measures what you actually care about: how much the
model's *generalisation* depends on that feature. Note that it splits credit oddly among
correlated features (shuffling one leaves the information available through the other).

In [ ]:
imp = pd.DataFrame({"feature": Xc.columns,
                    "impurity_importance": gs.best_estimator_.feature_importances_})

perm = permutation_importance(gs.best_estimator_, Xc_te, yc_te, n_repeats=30,
                             random_state=0, scoring="roc_auc")
imp["permutation_mean"] = perm.importances_mean
imp["permutation_sd"] = perm.importances_std
print(imp.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
o = imp.sort_values("impurity_importance")
ax[0].barh(o.feature, o.impurity_importance, color="steelblue")
ax[0].set_title("Impurity-based (training data, biased)")
o2 = imp.sort_values("permutation_mean")
ax[1].barh(o2.feature, o2.permutation_mean, xerr=o2.permutation_sd, color="seagreen")
ax[1].set_title("Permutation (test data, drop in ROC-AUC)")
plt.tight_layout(); plt.show()

In [ ]:
# The bias, demonstrated: add a pure-noise column with many distinct values
Xc2 = Xc_tr.copy()
Xc2["random_id"] = rng.random(len(Xc2))              # continuous noise, all values distinct
Xc2["random_binary"] = rng.integers(0, 2, len(Xc2))  # noise with only 2 values

t_bias = DecisionTreeClassifier(max_depth=None, random_state=0).fit(Xc2, yc_tr)
bias_imp = pd.DataFrame({"feature": Xc2.columns,
                         "impurity_importance": t_bias.feature_importances_}
                        ).sort_values("impurity_importance", ascending=False)
print(bias_imp.round(4).to_string(index=False))
print("\n'random_id' is pure noise, yet an unpruned tree ranks it among the most important")
print("features -- because a continuous column with 1,500 distinct values offers 1,499")
print("chances to carve out a lucky split. This is the cardinality bias.")

Xc2_te = Xc_te.copy()
Xc2_te["random_id"] = rng.random(len(Xc2_te))
Xc2_te["random_binary"] = rng.integers(0, 2, len(Xc2_te))
perm2 = permutation_importance(t_bias, Xc2_te, yc_te, n_repeats=20, random_state=0,
                               scoring="roc_auc")
print("\nPermutation importance on held-out data is not fooled:")
print(pd.DataFrame({"feature": Xc2.columns, "permutation": perm2.importances_mean.round(4)})
      .sort_values("permutation", ascending=False).to_string(index=False))

---
## 3.7 The geometry: axis-aligned boxes

Because every split tests **one feature against a threshold**, a tree can only draw
boundaries **parallel to the axes**. It partitions the feature space into rectangles.

This has two consequences:

- **Good:** interactions come for free. "High income *and* young" is two splits, no feature
  engineering required.
- **Bad:** a simple diagonal boundary needs a staircase of many splits to approximate — and
  each step is fitted from less data, so it is noisy.

The fix is not to abandon trees but to **ensemble** them (Notebook 6), or to rotate the
features (PCA) when you suspect diagonal structure.

In [ ]:
def boundary_plot(model, X2, y2, ax, title):
    h = 0.02
    x0, x1 = X2[:, 0].min()-0.5, X2[:, 0].max()+0.5
    y0, y1 = X2[:, 1].min()-0.5, X2[:, 1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x0, x1, h), np.arange(y0, y1, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="coolwarm", s=18, edgecolor="k", linewidth=0.3)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

# Case 1: a diagonal boundary -- hard for trees, trivial for a linear model
Xd = rng.normal(size=(300, 2))
yd = (Xd[:, 0] + Xd[:, 1] > 0).astype(int)
# Case 2: an axis-aligned box -- trivial for trees, impossible for a straight line
Xb_ = rng.uniform(-3, 3, size=(300, 2))
yb_ = ((np.abs(Xb_[:, 0]) < 1.5) & (np.abs(Xb_[:, 1]) < 1.5)).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, (Xs_, ys_, label) in enumerate([(Xd, yd, "diagonal"), (Xb_, yb_, "axis-aligned box")]):
    for col, (mdl, nm) in enumerate([
            (make_pipeline(StandardScaler(), LogisticRegression()), "logistic regression"),
            (DecisionTreeClassifier(max_depth=3, random_state=0), "tree, depth 3"),
            (DecisionTreeClassifier(max_depth=None, random_state=0), "tree, unlimited")]):
        mdl.fit(Xs_, ys_)
        acc = cross_val_score(mdl, Xs_, ys_, cv=SKF).mean()
        boundary_plot(mdl, Xs_, ys_, axes[row, col], f"{label}: {nm}\nCV accuracy {acc:.3f}")
plt.tight_layout(); plt.show()

print("Top row   : the tree needs a staircase for a boundary logistic regression draws")
print("            with one line -- and the staircase generalises worse.")
print("Bottom row: the tree nails the box; the linear model cannot represent it at all.")
print("\nNeither model is 'better'. They have different inductive biases.")

---
## 3.8 Strengths, weaknesses, and practical notes

**Strengths**

- **Interpretable** — a readable flowchart, and you can trace any single prediction
- **No scaling needed** — splits are order-based, so units are irrelevant
- **Mixed data types** — numeric and (encoded) categorical together
- **Non-linear and interaction-aware** for free
- **Robust to outliers in $x$** — a far-out value just falls on one side of a threshold
- Handles missing values natively in some implementations (LightGBM, XGBoost; scikit-learn's
  `HistGradientBoosting`)

**Weaknesses**

- **High variance** — small data changes give a very different tree
- **Overfits** without constraints
- **Cannot extrapolate**
- **Axis-aligned only** — diagonal structure is expensive
- **Greedy** — no guarantee of the best tree
- **Biased importances** toward high-cardinality features
- Struggles with **imbalanced** classes unless you set `class_weight`

**Practical notes**

- Do not scale. It changes nothing (see below).
- Encode categoricals as one-hot for scikit-learn; avoid arbitrary integer codes, which imply
  a false ordering.
- Always set `random_state` — ties are broken randomly, so trees are otherwise irreproducible.
- Use a single tree when you need explanation; use an ensemble when you need accuracy.

In [ ]:
# Trees are invariant to monotone transformations of the features
Xm = Xc_tr.copy()
Xm_scaled = pd.DataFrame(StandardScaler().fit_transform(Xm), columns=Xm.columns)
Xm_logged = Xm.assign(monthly=np.log(Xm.monthly), tenure=np.sqrt(Xm.tenure))

for name, Xuse in [("raw", Xm), ("standardised", Xm_scaled), ("log/sqrt transformed", Xm_logged)]:
    t = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xuse, yc_tr)
    print(f"  {name:<22} training accuracy {t.score(Xuse, yc_tr):.6f}, "
          f"{t.get_n_leaves()} leaves")
print("\nIdentical, because only the ORDER of feature values matters, and monotone")
print("transformations preserve order. Never waste a step scaling data for a tree.")

In [ ]:
# Instability: two trees on 90% subsamples of the same data can look completely different
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, seed in zip(axes, [1, 2]):
    idx = rng.choice(len(Xc_tr), int(0.9*len(Xc_tr)), replace=False)
    t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xc_tr.iloc[idx], yc_tr.iloc[idx])
    plot_tree(t, feature_names=list(Xc.columns), class_names=["stay", "churn"],
              filled=True, rounded=True, fontsize=7, ax=ax)
    ax.set_title(f"Trained on 90% subsample #{seed}", fontsize=10)
plt.tight_layout(); plt.show()

print("Different root splits, different thresholds, different structure -- from data that")
print("differs by 10% of rows. This is the HIGH VARIANCE of trees, and it is why:")
print("  * you should not over-interpret a single tree's exact structure")
print("  * averaging many trees (random forests) helps so much")

---
## Exercises

**Exercise 1.** Work out the first split by hand for the dataset below using **entropy**
instead of Gini, then confirm with scikit-learn. Does the criterion change the answer?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
d1 = pd.DataFrame({
    "hours_studied": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "attendance":    [60, 55, 90, 70, 95, 65, 98, 80, 99, 85],
    "buys":          [0, 0, 0, 0, 1, 0, 1, 1, 1, 1],
})
parent_H = entropy(d1.buys.value_counts(normalize=True))
print(f"Parent entropy = {parent_H:.4f} bits\n")

cands = []
for feat in ["hours_studied", "attendance"]:
    vals = np.sort(d1[feat].unique())
    for a, b in zip(vals[:-1], vals[1:]):
        g = split_gain(d1, feat, (a+b)/2, criterion=entropy)
        if g:
            cands.append(g)
c1 = pd.DataFrame(cands).sort_values("gain", ascending=False)
print("Top 5 splits by information gain (entropy):")
print(c1.head(5)[["feature", "threshold", "n_left", "n_right", "gain"]].round(4).to_string(index=False))

t_ent = DecisionTreeClassifier(criterion="entropy", random_state=0).fit(
    d1[["hours_studied", "attendance"]], d1.buys)
t_gini = DecisionTreeClassifier(criterion="gini", random_state=0).fit(
    d1[["hours_studied", "attendance"]], d1.buys)
names1 = ["hours_studied", "attendance"]
print(f"\nsklearn entropy root: {names1[t_ent.tree_.feature[0]]} <= "
      f"{t_ent.tree_.threshold[0]:.2f}")
print(f"sklearn gini    root: {names1[t_gini.tree_.feature[0]]} <= "
      f"{t_gini.tree_.threshold[0]:.2f}")
print(f"Our hand calculation: {c1.iloc[0].feature} <= {c1.iloc[0].threshold:.2f}")
print("\nSame answer. Gini and entropy rank splits almost identically -- they only differ")
print("in scale and curvature, not in which split wins.")

**Exercise 2.** Tune a decision tree on the wine dataset. Compare the default tree, a tuned
tree, and a `ccp_alpha`-pruned tree by cross-validated accuracy. Plot the final tree and
describe one rule in plain English.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
from sklearn.datasets import load_wine

wine = load_wine()
Xw, yw = pd.DataFrame(wine.data, columns=wine.feature_names), wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=0,
                                              stratify=yw)

default_tree = DecisionTreeClassifier(random_state=0)
tuned = GridSearchCV(DecisionTreeClassifier(random_state=0),
                     {"max_depth": [2, 3, 4, 5, None],
                      "min_samples_leaf": [1, 3, 5, 10]},
                     cv=SKF, scoring="accuracy").fit(Xw_tr, yw_tr)

pathw = DecisionTreeClassifier(random_state=0).fit(Xw_tr, yw_tr).cost_complexity_pruning_path(Xw_tr, yw_tr)
alphasw = [a for a in pathw.ccp_alphas[:-1] if a > 0]
cvw = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=0), Xw_tr, yw_tr,
                       cv=SKF).mean() for a in alphasw]
best_a = alphasw[int(np.argmax(cvw))]

for name, est in [("default (unconstrained)", default_tree),
                  ("tuned by grid search", tuned.best_estimator_),
                  (f"pruned, ccp_alpha={best_a:.4f}",
                   DecisionTreeClassifier(ccp_alpha=best_a, random_state=0))]:
    cvs = cross_val_score(est, Xw_tr, yw_tr, cv=SKF).mean()
    est.fit(Xw_tr, yw_tr)
    print(f"  {name:<32} CV acc {cvs:.4f}   test acc {est.score(Xw_te, yw_te):.4f}   "
          f"{est.get_n_leaves()} leaves")
print(f"\nBest grid-search parameters: {tuned.best_params_}")

In [ ]:
best_wine = DecisionTreeClassifier(**tuned.best_params_, random_state=0).fit(Xw_tr, yw_tr)
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(best_wine, feature_names=wine.feature_names, class_names=wine.target_names,
          filled=True, rounded=True, fontsize=8, ax=ax)
plt.title("Tuned wine classification tree")
plt.show()

print(export_text(best_wine, feature_names=list(wine.feature_names), decimals=2))
r_feat = wine.feature_names[best_wine.tree_.feature[0]]
r_thr = best_wine.tree_.threshold[0]
print(f"\nOne rule in plain English:")
print(f"  'If {r_feat} is at most {r_thr:.2f}, the wine is most likely NOT cultivar 2;")
print(f"   the tree then checks a second chemical measure to separate cultivars 0 and 1.'")
print("\nThat sentence is auditable by a chemist, which is the whole point of a tree.")

**Exercise 3.** Demonstrate the danger of impurity-based feature importance. Build a dataset
with one genuinely useful binary feature and one useless continuous feature with many distinct
values. Show that impurity importance prefers the useless one, and that permutation importance
does not.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
m3 = 1_200
useful_binary = rng.integers(0, 2, m3)
useless_continuous = rng.random(m3)                      # 1,200 distinct values, no signal
y3 = (rng.random(m3) < np.where(useful_binary == 1, 0.75, 0.25)).astype(int)

X3 = pd.DataFrame({"useful_binary": useful_binary, "useless_continuous": useless_continuous})
X3a, X3b, y3a, y3b = train_test_split(X3, y3, test_size=0.3, random_state=0, stratify=y3)

print(f"Truth: P(y=1 | binary=1) = 0.75, P(y=1 | binary=0) = 0.25")
print(f"       the continuous column is independent noise\n")

t3 = DecisionTreeClassifier(random_state=0).fit(X3a, y3a)      # unconstrained
p3 = permutation_importance(t3, X3b, y3b, n_repeats=30, random_state=0, scoring="roc_auc")

print("UNCONSTRAINED tree:")
print(pd.DataFrame({"feature": X3.columns,
                    "impurity_importance": t3.feature_importances_.round(4),
                    "permutation_importance": p3.importances_mean.round(4)}).to_string(index=False))
print(f"  training accuracy {t3.score(X3a, y3a):.4f}, test accuracy {t3.score(X3b, y3b):.4f}")
print("  -> impurity importance is dominated by the NOISE column, because it provided")
print("     hundreds of splits that carved out lucky pure leaves in training.")
print("  -> permutation importance on held-out data correctly gives it about zero.\n")

t3c = DecisionTreeClassifier(max_depth=2, min_samples_leaf=50, random_state=0).fit(X3a, y3a)
p3c = permutation_importance(t3c, X3b, y3b, n_repeats=30, random_state=0, scoring="roc_auc")
print("CONSTRAINED tree (max_depth=2, min_samples_leaf=50):")
print(pd.DataFrame({"feature": X3.columns,
                    "impurity_importance": t3c.feature_importances_.round(4),
                    "permutation_importance": p3c.importances_mean.round(4)}).to_string(index=False))
print(f"  test accuracy {t3c.score(X3b, y3b):.4f}  <- better, AND the importances are honest")
print("\nLesson: never read feature importances off an overfitted tree. Prune first,")
print("and prefer permutation importance measured on data the model has not seen.")

**Exercise 4 (challenge).** You must deliver a churn model that the compliance team can audit,
with each decision explainable in at most four questions. Build it, quantify what the
interpretability constraint costs in accuracy compared with an unconstrained tree and a
logistic regression, and write the recommendation.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

constrained = DecisionTreeClassifier(max_depth=4, min_samples_leaf=40, random_state=0)
unconstrained = DecisionTreeClassifier(random_state=0)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)

print(f"{'model':<34}{'CV AUC':>10}{'test AUC':>10}{'test acc':>10}{'explainable?':>14}")
for name, est, explain in [
        ("tree, depth<=4 (auditable)", constrained, "yes: <=4 rules"),
        ("tree, unconstrained", unconstrained, "no: 100s of rules"),
        ("logistic regression", logistic, "yes: coefficients"),
        ("random forest, 300 trees", forest, "no"),
]:
    cvv = cross_val_score(est, Xc_tr, yc_tr, cv=SKF, scoring="roc_auc").mean()
    est.fit(Xc_tr, yc_tr)
    pb = est.predict_proba(Xc_te)[:, 1]
    print(f"{name:<34}{cvv:>10.4f}{roc_auc_score(yc_te, pb):>10.4f}"
          f"{accuracy_score(yc_te, est.predict(Xc_te)):>10.4f}{explain:>14}")

In [ ]:
constrained.fit(Xc_tr, yc_tr)
print("The deliverable -- the full rulebook, in plain text:\n")
print(export_text(constrained, feature_names=list(Xc.columns), decimals=1))

print("Sanity check: the deepest path really is 4 questions long.")
print(f"  tree depth = {constrained.get_depth()}, leaves = {constrained.get_n_leaves()}\n")

auc_c = roc_auc_score(yc_te, constrained.predict_proba(Xc_te)[:, 1])
forest.fit(Xc_tr, yc_tr)
auc_f = roc_auc_score(yc_te, forest.predict_proba(Xc_te)[:, 1])
logistic.fit(Xc_tr, yc_tr)
auc_l = roc_auc_score(yc_te, logistic.predict_proba(Xc_te)[:, 1])

print("RECOMMENDATION")
print(f"  The auditable tree reaches test AUC {auc_c:.4f} against {auc_f:.4f} for the")
print(f"  random forest -- a cost of {auc_f - auc_c:.4f} AUC ({(auc_f-auc_c)/auc_f*100:.1f}%)")
print(f"  for full explainability. Logistic regression sits at {auc_l:.4f} and is also")
print("  auditable, but its explanation is a set of coefficients rather than a rulebook,")
print("  and it cannot express the interactions compliance keeps asking about.")
print()
print("  I would ship the depth-4 tree as the decision system, and run the random forest")
print("  in shadow mode. If the shadow model's advantage grows materially, that is the")
print("  evidence needed to negotiate a more complex model with a surrogate explanation")
print("  layer -- rather than trading away auditability on a guess.")
print()
print("  Note also what the constraint bought us besides compliance: the unconstrained")
print(f"  tree scores {roc_auc_score(yc_te, unconstrained.fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:,1]):.4f},")
print("  WORSE than the constrained one. Interpretability and accuracy are not always")
print("  in conflict -- here the depth limit is also the regularisation.")

---
## Summary

| Concept | Key point |
|---|---|
| Splitting | Greedy: try every feature and threshold, maximise information gain |
| Gini | $1-\sum p_k^2$ — default, no logarithm |
| Entropy | $-\sum p_k\log_2 p_k$ — nearly identical results |
| Information gain | Parent impurity minus the sample-weighted child impurity |
| Leaf prediction | Majority class (classification) or mean (regression) |
| Regression trees | Minimise squared error; produce a **staircase**; cannot extrapolate |
| Overfitting | Unconstrained trees memorise; training accuracy hits 1.0 |
| Pre-pruning | `max_depth`, `min_samples_leaf`, `max_leaf_nodes` |
| Post-pruning | `ccp_alpha`, chosen by cross-validation |
| Feature importance | Impurity-based is biased toward high-cardinality; prefer permutation |
| Geometry | Axis-aligned rectangles: interactions free, diagonals expensive |
| No scaling needed | Splits depend only on order |
| High variance | Small data changes give very different trees → ensemble them |

**Next up:** [Notebook 4 — K-Nearest Neighbors](4.%20K-Nearest%20Neighbors.ipynb), a model
with no training phase at all.